# 90 · Sealed Evaluation — Day 650

**Purpose.** One honest number. Day 650 has never been touched: no model
trained on it, no hyperparameter chosen by it, no feature inspected against
it. This notebook opens it once.

## What is being compared

The candidate pool is held **fixed** — every model receives the identical
~544 candidates per household. Only the *scoring rule* changes:

| Rule | Score | What it represents |
|---|---|---|
| popularity | item_baskets_365d | the agnostic base rate |
| recency (buy-again) | −days_since_last | the champion heuristic |
| als_affinity | learned embedding dot product | representation learning alone |
| **xgb_ranker** | trained model | all signals fused |

Holding the pool fixed isolates exactly what stage two contributes, which the
earlier notebooks could not do (each model had its own candidate set).

## Beyond the headline

1. **Segment cuts** — recall by household activity tertile. An average can
   hide a model that works only for heavy shoppers.
2. **Revenue-weighted recall** — hits weighted by the dollars actually spent
   on that item. Recall counts items; the business counts money, and this
   dataset's #2 item by volume (fuel) is its #1 by revenue.

## Discipline note

The test set is spent after this notebook. Any model built later is
validation-only (day 600) and will be labeled as such — re-opening day 650
for a newer model would be selecting on the test set.

In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import json
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import yaml
from pathlib import Path

from retail_ds.features.design_matrix import build_plane
from retail_ds.evaluate.metrics import recall_at_k, hit_rate_at_k, ndcg_at_k

ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parents[1]

CFG = yaml.safe_load((ROOT / "configs" / "base.yaml").read_text())
K = CFG["top_k"]
TEST_DAY = CFG["snapshots"]["test"]      # 650
HORIZON = CFG["label_horizon_days"]
BUDGETS = {k: (10**9 if v == -1 else v) for k, v in CFG["candidates"].items()}

con = duckdb.connect((ROOT / "db" / "retail.duckdb").as_posix(), read_only=True)

REG = ROOT / "models" / "registry"
meta = json.loads((REG / "model_meta.json").read_text())
model = xgb.XGBClassifier()
model.load_model(REG / "ranker.ubj")
FEATURES = meta["features"]

sns.set_theme(style="whitegrid", context="notebook")
print("test day:", TEST_DAY, "| model trained on planes:", meta["trained_on_planes"])
print("valid recall@10 was:", meta["valid_recall_at_10"])

C:\Users\siava\Recommendation_system\retail-recsys-platform\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


test day: 650 | model trained on planes: [450, 510, 570]
valid recall@10 was: 0.104


In [2]:
test = build_plane(con, TEST_DAY, BUDGETS, horizon=HORIZON, with_label=True)
labels = con.sql(
    f"SELECT household_key, product_id FROM purchase_labels({TEST_DAY}, {HORIZON})"
).df()

n_bought = labels.groupby("household_key").size()
oracle = (n_bought.clip(upper=K) / n_bought).mean()
print(f"pool {test.shape} | labels {len(labels):,} | oracle ceiling {oracle:.3f}")

100%|██████████| 20/20 [00:05<00:00,  3.57it/s]


pool (1451856, 26) | labels 107,971 | oracle ceiling 0.396


In [3]:
test["xgb_score"] = model.predict_proba(test[FEATURES])[:, 1]
test["recency_score"] = -test["days_since_last"]

def top_k(df, score_col):
    d = df[["household_key", "product_id", score_col]].copy()
    d["rank"] = (d.groupby("household_key")[score_col]
                 .rank(method="first", ascending=False, na_option="bottom").astype(int))
    return d[d["rank"] <= K]

rules = {
    "popularity":   "item_baskets_365d",
    "recency":      "recency_score",
    "als_affinity": "als_affinity",
    "xgb_ranker":   "xgb_score",
}
recs = {name: top_k(test, col) for name, col in rules.items()}

sealed = pd.DataFrame([{
    "model": name,
    f"recall@{K}":   recall_at_k(r, labels, K),
    f"hit_rate@{K}": hit_rate_at_k(r, labels, K),
    f"ndcg@{K}":     ndcg_at_k(r, labels, K),
} for name, r in recs.items()]).set_index("model").round(3)
sealed["share_of_ceiling"] = (sealed[f"recall@{K}"] / oracle).round(2)
sealed.sort_values(f"recall@{K}", ascending=False)

,recall@10,hit_rate@10,ndcg@10,share_of_ceiling
model,,,,
xgb_ranker,0.105,0.865,0.442,0.26
recency,0.052,0.683,0.181,0.13
popularity,0.045,0.722,0.204,0.11
als_affinity,0.029,0.405,0.071,0.07


In [4]:
def top_k_sort(df, cols, ascending):
    d = df[["household_key", "product_id"] + cols].copy()
    d = d.sort_values(["household_key"] + cols, ascending=[True] + ascending)
    d["rank"] = d.groupby("household_key").cumcount() + 1
    return d[d["rank"] <= K]

recs["buy_again"] = top_k_sort(test, ["days_since_last", "times_bought"], [True, False])
recs.pop("recency", None)

sealed = pd.DataFrame([{
    "model": name,
    f"recall@{K}":   recall_at_k(r, labels, K),
    f"hit_rate@{K}": hit_rate_at_k(r, labels, K),
    f"ndcg@{K}":     ndcg_at_k(r, labels, K),
} for name, r in recs.items()]).set_index("model").round(3)
sealed["share_of_ceiling"] = (sealed[f"recall@{K}"] / oracle).round(2)
sealed = sealed.sort_values(f"recall@{K}", ascending=False)
sealed.to_csv(ROOT / "reports" / "sealed_test_day650.csv")
sealed

,recall@10,hit_rate@10,ndcg@10,share_of_ceiling
model,,,,
xgb_ranker,0.105,0.865,0.442,0.26
buy_again,0.066,0.750,0.284,0.17
popularity,0.045,0.722,0.204,0.11
als_affinity,0.029,0.405,0.071,0.07


In [5]:
def recall_per_household(r, labs, k=K):
    hits = r[r["rank"] <= k].merge(labs, on=["household_key", "product_id"])
    n_rel = labs.groupby("household_key").size()
    return (hits.groupby("household_key").size() / n_rel).reindex(n_rel.index).fillna(0)

activity = (test.groupby("household_key")["baskets_90d"].first())
segment = pd.qcut(activity, 3, labels=["light", "medium", "heavy"])

seg_table = pd.DataFrame({
    name: recall_per_household(r, labels).groupby(segment).mean()
    for name, r in recs.items()
}).round(3)
seg_table

C:\Users\siava\AppData\Local\Temp\ipykernel_20548\1314770794.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  name: recall_per_household(r, labels).groupby(segment).mean()
C:\Users\siava\AppData\Local\Temp\ipykernel_20548\1314770794.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  name: recall_per_household(r, labels).groupby(segment).mean()
C:\Users\siava\AppData\Local\Temp\ipykernel_20548\1314770794.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the f

,popularity,als_affinity,xgb_ranker,buy_again
baskets_90d,,,,
light,0.055,0.053,0.112,0.076
medium,0.049,0.032,0.116,0.073
heavy,0.036,0.011,0.090,0.054


In [6]:
label_value = con.sql(f"""
    SELECT household_key, product_id, SUM(sales_value) AS value
    FROM staging.stg_transactions
    WHERE day_no > {TEST_DAY} AND day_no <= {TEST_DAY} + {HORIZON}
    GROUP BY household_key, product_id
""").df()
total_value = label_value["value"].sum()

rev = {}
for name, r in recs.items():
    hits = r.merge(label_value, on=["household_key", "product_id"])
    rev[name] = hits["value"].sum() / total_value
pd.Series(rev, name="revenue_weighted_recall@10").sort_values(ascending=False).round(3)

xgb_ranker      0.158
buy_again       0.085
popularity      0.080
als_affinity    0.015
Name: revenue_weighted_recall@10, dtype: float64

In [7]:
per_hh_ceiling = (n_bought.clip(upper=K) / n_bought)
pd.DataFrame({
    "oracle_ceiling": per_hh_ceiling.groupby(segment).mean(),
    "xgb_recall": recall_per_household(recs["xgb_ranker"], labels).groupby(segment).mean(),
}).assign(share_of_ceiling=lambda d: (d.xgb_recall / d.oracle_ceiling)).round(3)

C:\Users\siava\AppData\Local\Temp\ipykernel_20548\4082721858.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  "oracle_ceiling": per_hh_ceiling.groupby(segment).mean(),
C:\Users\siava\AppData\Local\Temp\ipykernel_20548\4082721858.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  "xgb_recall": recall_per_household(recs["xgb_ranker"], labels).groupby(segment).mean(),


,oracle_ceiling,xgb_recall,share_of_ceiling
baskets_90d,,,
light,0.616,0.112,0.182
medium,0.436,0.116,0.267
heavy,0.222,0.090,0.408


Segments (household activity tertiles): ranker leads in all three on raw
recall — light 0.112 vs 0.076, medium 0.116 vs 0.073, heavy 0.090 vs 0.054.
But raw recall is confounded by basket breadth: per-segment oracle ceilings
are 0.616 / 0.436 / 0.222. Normalized, the ordering **reverses** — the model
achieves 18.2% / 26.7% / 40.8% of ceiling. Heavy households (deep history)
are the model's strongest segment; **light households are the true weakness**
— thin history, near-cold-start, where popularity/ALS candidates and
content-based features would pay. Raw recall would have sent optimization
effort to exactly the wrong segment.